## **PORTAFOLIO ACADÉMICO DE BI:**
# **PROYECTO INMOBILIARIO — CIUDAD DE LA COSTA**
# **Equipamiento — Valor agregado**
---
## Módulo 4:
* Introducción
* Conexión y recuperación de Datos
* Preguntas empresariales:
1. ¿Cuánto suma cada *amenity* al precio?
2. Combinación de amenities: Tahona y Solymar
3. Precio según habitaciones y baños
4. Ratio terreno/edificado por barrio
5. Tipo de construcción vs. precio
6. Complejos y barrios privados vs. vivienda independiente

---
## **Introducción**
Con la geografía ya explicada este módulo mide cuánto "suma" al precio cada característica física de la propiedad: *amenities* estructurados (piscina, cochera, parrillero), cantidad de ambientes, proporción terreno/construcción, y detalles.

El objetivo es cuantificar el valor agregado de cada atributo más allá de la ubicación.

**Recordatorio de revisión de Metadatos y Gobernanza:** Todos los hallazgos de este Módulo siguen sujetos a los mismos límites metodológicos ya documentados por lo que cada conclusión de negocio debe leerse con ese contexto de fondo.

---

### **Conexión y recuperación de Datos del Módulo 1:**

Para dar inicio al **Módulo 4** debemos vincular el cuaderno de trabajo con la infraestructura de datos que ya dejamos consolidada.

In [ ]:
# Autenticamos la cuenta de Google en este nuevo cuaderno
from google.colab import auth
auth.authenticate_user()

print("Autenticación completa")

Autenticación completa


In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

# 1. Nos autenticamos asegurando el proyecto destino
auth.authenticate_user(project_id="proyectosuy")

# 2. Inicializamos el cliente pasándole explícitamente el ID en el constructor
client = bigquery.Client(project="proyectosuy")

# 3. Definimos la consulta
query_toda_la_base = """
    SELECT *
    FROM `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
"""

# 4. Cargamos todo el Data Frame completo
# Pasamos el project_id también en el método de ejecución por seguridad
df_completo = client.query(query_toda_la_base, project="proyectosuy").to_dataframe()

# Verificamos que cargaron todas las columnas y filas
print(f"Base de datos cargada. Total de propiedades: {len(df_completo)}")
df_completo.head()

Base de datos cargada. Total de propiedades: 555


,id,publicacion,finalizacion,operacion,tipo_inmueble,moneda,precio,ubicacion,zona,pisos,...,banos,cochera,parrillero,jardin,piscina,mts2_terreno,mts2_edificado,antiguedad,gastos_comunes_UYU,detalles
0,1,2025-06-01,NaT,Venta,Casa,U$S,289000.0,Solymar,Sur,2,...,4,1,True,True,False,364,139.0,20,NaN,Construcción sólida
1,2,2026-05-07,NaT,Venta,Casa,U$S,250000.0,Solymar,Sur,1,...,2,1,True,True,False,310,95.0,1,NaN,Construcción sólida
2,3,2026-05-05,NaT,Venta,Complejo,U$S,187000.0,Solymar,Norte,2,...,2,2,True,False,False,148,78.2,0,NaN,Próximo a Car One
3,4,2026-01-12,NaT,Venta,Casa,U$S,265000.0,Solymar,Sur,1,...,2,1,True,True,False,527,210.0,30,NaN,Casa independiente
4,5,2026-04-02,NaT,Venta,Casa,U$S,185000.0,Lagomar,Norte,1,...,1,1,True,True,False,200,85.0,1,NaN,Próximo a Almenara Mall


# **Preguntas empresariales:**

### **Pregunta 1. ¿Cuánto suma cada *amenity* al precio?**
Buscamos cuantificar el premio de precio que representa tener piscina, cochera o parrillero comparando el precio promedio de las viviendas que tienen cada *amenity* contra las que no lo tienen.

Lo hacemos controlando por barrio porque sabemos por los Módulos anteriores que el precio varía muchísimo según ubicación.

In [ ]:
# =============================================================================
# PREMIO DE PRECIO POR AMENITY, CONTROLANDO POR BARRIO
# =============================================================================

# Para cada barrio, calculamos el precio promedio de venta de vivienda
# separando las propiedades CON y SIN cada amenity (cochera, parrillero,
# piscina), para poder comparar el premio de precio dentro del mismo barrio.
query_premio_amenities = """
SELECT
    ubicacion AS barrio,

    -- ===== COCHERA =====
    COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
            AND LOWER(operacion) = 'venta' AND cochera > 0) AS cantidad_con_cochera,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta' AND cochera > 0
               THEN precio END), 0) AS precio_promedio_con_cochera,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta' AND (cochera = 0 OR cochera IS NULL)
               THEN precio END), 0) AS precio_promedio_sin_cochera,

    -- ===== PARRILLERO =====
    COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
            AND LOWER(operacion) = 'venta' AND parrillero = TRUE) AS cantidad_con_parrillero,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta' AND parrillero = TRUE
               THEN precio END), 0) AS precio_promedio_con_parrillero,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta' AND parrillero = FALSE
               THEN precio END), 0) AS precio_promedio_sin_parrillero,

    -- ===== PISCINA =====
    COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
            AND LOWER(operacion) = 'venta' AND piscina = TRUE) AS cantidad_con_piscina,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta' AND piscina = TRUE
               THEN precio END), 0) AS precio_promedio_con_piscina,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta' AND piscina = FALSE
               THEN precio END), 0) AS precio_promedio_sin_piscina
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
GROUP BY
    barrio
ORDER BY
    barrio;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_premio_amenities = client.query(query_premio_amenities).to_dataframe()

# Desplegamos el resultado.
df_premio_amenities

,barrio,cantidad_con_cochera,precio_promedio_con_cochera,precio_promedio_sin_cochera,cantidad_con_parrillero,precio_promedio_con_parrillero,precio_promedio_sin_parrillero,cantidad_con_piscina,precio_promedio_con_piscina,precio_promedio_sin_piscina
0,Carrasco,16,291131.0,380100.0,17,271971.0,483775.0,4,605900.0,243235.0
1,Lagomar,25,294320.0,NaN,23,295565.0,280000.0,2,592500.0,268391.0
2,Pinar,16,250938.0,201500.0,11,276727.0,196286.0,4,287500.0,233429.0
3,Shangrila,27,299556.0,316000.0,29,314276.0,184667.0,5,373800.0,288852.0
4,Solymar,158,280607.0,252128.0,162,280365.0,248571.0,27,368237.0,260876.0
5,Tahona,26,659500.0,588500.0,29,652103.0,590000.0,21,658857.0,629444.0


### **Diagnóstico: Premio de precio por amenity**

**1. Piscina:** el premio más consistente y fuerte de los tres *amenities*, en todos los barrios tener piscina se asocia a un precio más alto sin una sola excepción. El caso más marcado es Carrasco: USD 605,900 (con piscina) vs. USD 243,235 (sin piscina) una diferencia de 149%.

**2. Parrillero:** también muestra premio positivo en casi todos los barrios, con Shangrilá como el caso más extremo: USD 314,276 (con) y USD 184,667 (sin), una diferencia de 70%. Esto es llamativo porque el parrillero no suena como un *amenity* de tanto peso, probablemente esté funcionando como un "proxy" de vivienda más completa y de mayor categoría en general, no como el parrillero en sí explicando semejante diferencia de precio.

**3. Cochera:** es el amenity más inconsistente, en algunos barrios el premio se invierte. En Carrasco y Shangrilá las propiedades sin cochera son más caras que las que la tienen (Carrasco: USD 380,100 sin y USD 291,131 con). Esto es contraintuitivo y probablemente se explica por composición de tipo de inmueble: es posible que las propiedades sin cochera en esos barrios sean predominantemente terrenos grandes o casas de mayor categoría con otras características que compensan más que la cochera "restando" valor.

**4. Piscina sigue siendo el amenity con menor tamaño de muestra** "con" en la mayoría de los barrios (Carrasco 4, Lagomar 2, Pinar 4, Shangrilá 5), excepto en La Tahona (21) y Solymar (27). Cualquier premio de precio de piscina en esos barrios chicos debe leerse como orientativo, no concluyente.

**Advertencia crítica de calidad de datos:** Lagomar no tiene ningún caso "sin cochera", la columna precio_promedio_sin_cochera de Lagomar da NaN, no hay una sola propiedad en ese barrio sin cochera dentro de la muestra filtrada así que no se puede calcular ningún premio para ese *amenity* ahí.

---
### **Pregunta 2. Combinación de amenities: Tahona y Solymar**
Ya vimos el premio de cada amenity por separado, ahora buscamos entender el perfil combinado: ¿cuántos amenities suele tener una propiedad típica en el segmento premium frente a el masivo? No es lo mismo tener piscina sola que piscina + cochera + parrillero, esta pregunta mide qué tan "completo" es el paquete de confort en cada segmento.

In [ ]:
# =============================================================================
# COMBINACIÓN DE AMENITIES — TAHONA vs. SOLYMAR
# =============================================================================

# Para cada vivienda individual de Tahona y Solymar, contamos cuántos
# de los 3 amenities principales tiene (0 a 3), y después vemos cómo
# se distribuye esa "cantidad de amenities" en cada barrio.
query_combinacion_amenities = """
WITH amenities_por_propiedad AS (
    SELECT
        ubicacion AS barrio,
        precio,

        -- Sumamos 1 por cada amenity presente, para obtener un score
        -- de 0 a 3 que representa qué tan "completa" está la propiedad.
        (CASE WHEN cochera > 0 THEN 1 ELSE 0 END)
        + (CASE WHEN parrillero = TRUE THEN 1 ELSE 0 END)
        + (CASE WHEN piscina = TRUE THEN 1 ELSE 0 END) AS cantidad_amenities
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        ubicacion IN ('Tahona', 'Solymar')
        AND LOWER(tipo_inmueble) != 'terreno'
        AND LOWER(moneda) = 'u$s'
        AND LOWER(operacion) = 'venta'
)

# Agrupamos por barrio y cantidad de amenities, para ver la distribución
# de "paquetes de confort" en cada segmento.
SELECT
    barrio,
    cantidad_amenities,
    COUNT(*) AS cantidad_propiedades,

    -- Porcentaje que representa este nivel de amenities dentro del barrio.
    ROUND(
        SAFE_DIVIDE(
            COUNT(*),
            SUM(COUNT(*)) OVER (PARTITION BY barrio)
        ) * 100, 1
    ) AS porcentaje_del_barrio,

    ROUND(AVG(precio), 0) AS precio_promedio_usd
FROM
    amenities_por_propiedad
GROUP BY
    barrio, cantidad_amenities
ORDER BY
    barrio, cantidad_amenities DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_combinacion_amenities = client.query(query_combinacion_amenities).to_dataframe()

# Desplegamos el resultado.
df_combinacion_amenities

,barrio,cantidad_amenities,cantidad_propiedades,porcentaje_del_barrio,precio_promedio_usd
0,Solymar,3,25,13.7,380096.0
1,Solymar,2,118,64.5,265530.0
2,Solymar,1,36,19.7,236811.0
3,Solymar,0,4,2.2,319750.0
4,Tahona,3,19,63.3,659316.0
5,Tahona,2,8,26.7,667375.0
6,Tahona,1,3,10.0,545000.0


### **Diagnóstico: Combinación de amenities**

**1. La Tahona es el segmento de "paquete completo":** 63.3% de sus viviendas tienen los 3 *amenities* y ninguna propiedad de La Tahona tiene 0 *amenities*, el mínimo en ese barrio es 1. Esto confirma con números el perfil premium que veníamos describiendo desde el Módulo 1: comprar en La Tahona no es solo pagar por ubicación o tamaño, es comprar un estándar de confort ya consolidado como norma del barrio.

**2. Solymar tiene un perfil mucho más disperso:** con el "combo de 2 *amenities*" como el más común (64.5%), el paquete completo (3 amenities) es minoritario (13.7%) y hay un 2.2% de propiedades sin ningún *amenity* de los tres. Esto confirma que Solymar es un mercado más heterogéneo en términos de equipamiento coherente con su perfil de "mercado masivo" que ya veníamos viendo.

**3. Dentro de La Tahona el precio no sube de forma lineal con la cantidad de amenities:** Las propiedades con 2 *amenities* (USD 667,375) son incluso más caras en promedio que las de 3 (USD 659,316), una diferencia chica pero que refuerza que en el segmento premium el precio está determinado por muchos factores combinados (tamaño, terreno, ubicación específica) y no simplemente por sumar *amenities* uno a uno.

**Dato curioso:** en Solymar tener 0 *amenities* es más caro (USD 319,750) que tener 1 (USD 236,811), esto contradice la lógica simple de "más *amenities* = más precio" y sugiere que esas 4 propiedades sin ningún *amenity* probablemente son grandes terrenos o casas atípicas (quizás de mayor tamaño o mejor ubicación puntual) que compensan la falta de equipamiento con otro atributo de valor. Vale la pena marcarlo como una anomalía a no sobre-interpretar, dado que es una base de solo 4 propiedades.

---

### **Pregunta 3. Precio según habitaciones y baños**
Buscamos ver cómo varía el precio de venta según la cantidad de habitaciones y de baños dentro de cada barrio, esto le sirve a un comprador o inversor para estimar un "precio esperado" según la composición de la propiedad, en vez de guiarse solo por el promedio general del barrio.

In [ ]:
# =============================================================================
# PRECIO SEGÚN HABITACIONES Y BAÑOS
# =============================================================================

# Calculamos el precio promedio de venta, segmentado por barrio y por
# cantidad de habitaciones, para ver cómo escala el precio dentro de
# cada barrio a medida que aumentan los ambientes.
query_precio_habitaciones = """
SELECT
    ubicacion AS barrio,
    habitaciones,

    -- Cantidad de propiedades consideradas en este cruce barrio + habitaciones.
    COUNT(*) AS cantidad_propiedades,

    ROUND(AVG(precio), 0) AS precio_promedio_usd,

    -- Precio por m2 edificado, para separar el efecto "más habitaciones
    -- = más metros" del efecto "más habitaciones = mejor categoría".
    ROUND(AVG(CASE WHEN mts2_edificado IS NOT NULL AND mts2_edificado > 0
               THEN SAFE_DIVIDE(precio, mts2_edificado) END), 0) AS precio_x_m2_promedio_usd
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
WHERE
    LOWER(tipo_inmueble) != 'terreno'
    AND LOWER(moneda) = 'u$s'
    AND LOWER(operacion) = 'venta'
    AND habitaciones IS NOT NULL
GROUP BY
    barrio, habitaciones
HAVING
    -- Filtramos combinaciones con muy pocos casos, para no reportar
    -- promedios poco confiables con 1 o 2 propiedades.
    cantidad_propiedades >= 3
ORDER BY
    barrio, habitaciones;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_precio_habitaciones = client.query(query_precio_habitaciones).to_dataframe()

# Desplegamos el resultado.
df_precio_habitaciones

,barrio,habitaciones,cantidad_propiedades,precio_promedio_usd,precio_x_m2_promedio_usd
0,Carrasco,2,5,146400.0,1927.0
1,Carrasco,3,10,325500.0,2944.0
2,Carrasco,4,4,467525.0,2864.0
3,Lagomar,2,9,201778.0,2457.0
4,Lagomar,3,12,289000.0,2670.0
5,Lagomar,4,3,374667.0,3406.0
6,Pinar,2,4,132750.0,2056.0
7,Pinar,3,10,227900.0,1599.0
8,Pinar,4,4,402000.0,1323.0
9,Shangrila,2,7,199857.0,2522.0


### **Diagnóstico: Precio según habitaciones**

**1. El patrón general es consistente:** A más habitaciones, más precio total en todos los barrios sin excepción. Esto es esperable pero confirma que la relación se sostiene de forma pareja en toda la Costa, sin *outliers* de barrio que la rompan.+

**2. El valor por m2 NO sigue el mismo patrón:** A diferencia del precio total el precio por metro cuadrado no crece de forma consistente con más habitaciones. Ejemplos claros:

Pinar: 2 hab = USD 2,056/m2 pero 4 hab = solo USD 1,323/m2, una caída del 36%.

Solymar: 3 hab = USD 2,397/m2 pero 4 hab = USD 1,710/m2, cae 29%.

Shangrilá: 3 hab = USD 2,717/m2 pero 4 hab = USD 1,562/m2, cae 43%.

Este es el mismo fenómeno de economía de escala que ya habíamos detectado en el Módulo 2 (Pregunta 6, correlación negativa entre tamaño y valor por m2), ahora lo estamos viendo confirmado también a través de la cantidad de habitaciones no solo de los metros cuadrados en sí.

Comprar una propiedad más grande (más habitaciones) sale proporcionalmente más barato por metro.

**3. La Tahona es la excepción:** El valor por m2 se mantiene estable e incluso sube levemente con más habitaciones. 3 habitaciones (USD 2,550/m2) 4 habitaciones (USD 2,730/m2) 5 habitaciones (USD 2,451/m2), sin la caída pronunciada que vemos en el resto de los barrios. Esto es coherente con lo que ya sabíamos: en el segmento premium el tamaño no "diluye" el valor por metro de la misma forma, porque el precio está más determinado por ubicación y categoría general que por pura superficie.

**4. La Tahona también es el único barrio sin propiedades chicas:** En la muestra filtrada el mínimo es 3 habitaciones, reforzando su perfil de vivienda familiar de mayor tamaño desde la base.

**5. Solymar es el barrio con la escala más completa:** Gracias a su gran volumen de stock es el único que permite ver el patrón completo de principio a fin sin cortes por falta de muestra.

---

### **Pregunta 4. Ratio terreno/edificado por barrio (lotes grandes vs. construcción compacta)**
Buscamos entender, para las viviendas ya construidas de cada barrio, qué proporción del terreno está efectivamente construida.
Un ratio alto (terreno mucho más grande que lo edificado) indica lotes amplios con construcción compacta, típico de casas con jardín grande o potencial de ampliación.
Un ratio bajo (terreno similar a lo edificado) indica construcción densa con poco espacio libre, más típico de propiedades urbanas o de aprovechamiento máximo del lote.

In [ ]:
# =============================================================================
# RATIO TERRENO/EDIFICADO POR BARRIO
# =============================================================================

query_ratio_terreno_edificado = """
WITH ratio_individual AS (
    SELECT
        ubicacion AS barrio,
        mts2_terreno,
        mts2_edificado,

        -- Ratio de esta propiedad puntual: cuántas veces más grande es
        -- el terreno respecto a lo edificado. Un ratio de 2.0 significa
        -- que el terreno es el doble de grande que la construcción.
        SAFE_DIVIDE(mts2_terreno, mts2_edificado) AS ratio_terreno_edificado
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        LOWER(tipo_inmueble) != 'terreno'
        AND mts2_terreno IS NOT NULL
        AND mts2_terreno > 0
        AND mts2_edificado IS NOT NULL
        AND mts2_edificado > 0
)

# Agregamos por barrio para obtener el ratio promedio y mediano,
# junto con la cantidad de propiedades consideradas.
SELECT
    barrio,
    COUNT(*) AS cantidad_propiedades,
    ROUND(AVG(ratio_terreno_edificado), 2) AS ratio_promedio,
    ROUND(APPROX_QUANTILES(ratio_terreno_edificado, 2)[OFFSET(1)], 2) AS ratio_mediano,

    -- Porcentaje del terreno que está efectivamente construido (inverso
    -- del ratio), una forma más intuitiva de leer el mismo dato:
    -- un valor bajo significa mucho terreno libre sin construir.
    ROUND(AVG(SAFE_DIVIDE(mts2_edificado, mts2_terreno)) * 100, 1) AS porcentaje_terreno_construido
FROM
    ratio_individual
GROUP BY
    barrio
ORDER BY
    ratio_promedio DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_ratio_terreno_edificado = client.query(query_ratio_terreno_edificado).to_dataframe()

# Desplegamos el resultado.
df_ratio_terreno_edificado

,barrio,cantidad_propiedades,ratio_promedio,ratio_mediano,porcentaje_terreno_construido
0,Tahona,32,4.12,3.21,34.1
1,Pinar,21,4.06,3.33,33.0
2,Solymar,214,3.39,2.82,39.4
3,Carrasco,18,3.22,2.49,45.7
4,Lagomar,30,2.81,2.35,43.5
5,Shangrila,44,2.65,2.49,45.1


### **Diagnóstico: Ratio terreno/edificado por barrio**

**1. La Tahona tiene los lotes proporcionalmente más grandes de toda la Costa.** Con un ratio promedio de 4.12 (el terreno es en promedio, más de 4 veces más grande que lo construido) y solo un 34.1% del terreno efectivamente edificado, confirma el perfil de baja densidad que veníamos viendo: casas grandes en lotes aún más grandes, con mucho espacio libre (jardín, patio, distancia entre construcciones).

**2. El Pinar sorprende en segundo lugar.** Ratio de 4.06, muy cerca de La Tahona, es un dato interesante porque contrasta con el perfil "económico" que veníamos asociando a este barrio. Tiene lotes proporcionalmente casi tan grandes como el segmento premium aunque con construcciones de menor valor. Esto sugiere que en El Pinar hay terreno disponible en abundancia relativa.

**3. Shangrilá y Lagomar tienen la construcción más densa.** Más de 43% del terreno edificado, con ratios de 2.65 y 2.81 respectivamente, son los barrios donde las propiedades aprovechan proporcionalmente más su lote dejando menos espacio libre, un perfil más urbano/compacto dentro del conjunto.

**4. Media y mediana muestran una brecha considerable en todos los barrios.** Por ejemplo en La Tahona: 4.12 vs 3.21, señal de que también acá hay algunos lotes extra grandes tirando el promedio hacia arriba. Es el mismo patrón de asimetría que venimos viendo en casi toda métrica de este proyecto.

Vale la pena leer el ratio mediano como el valor más representativo del "lote típico" de cada barrio, no el promedio.


**5. Solymar tiene un ratio intermedio.** (3.39 media / 2.82 mediana)Ubicándose entre el perfil de baja densidad de La Tahona/Pinar y el más compacto de Shangrilá/Lagomar coherente con su rol de "mercado masivo y diverso" que ya conocemos.

---

### **Pregunta 5. Tipo de construcción vs. precio**
Hasta ahora usamos detalles solo para proximidad geográfica, esta pregunta explota una parte distinta de esa misma columna: las categorías que describen el tipo de construcción (Chalet, Casa contenedor, Construcción sólida, Casa de piedra, Cabaña, etc).

Buscamos ver si el estilo constructivo declarado se asocia a un premio o descuento de precio más allá del barrio y del tamaño.

In [ ]:
# =============================================================================
# TIPO DE CONSTRUCCIÓN vs. PRECIO
# =============================================================================

# Filtramos solo los "detalles" que describen tipo/estilo de construcción
# (dejando afuera proximidad y amenities puntuales), y calculamos precio
# promedio y $/m2 para cada categoría constructiva.
query_precio_tipo_construccion_completo = """
SELECT
    detalles,

    -- Cantidad de viviendas en venta consideradas para esta categoría.
    COUNTIF(LOWER(tipo_inmueble) != 'terreno'
            AND LOWER(moneda) = 'u$s'
            AND LOWER(operacion) = 'venta') AS cantidad_propiedades,

    -- Precio promedio de venta, en USD.
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                    AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta'
               THEN precio END), 0) AS precio_promedio_usd,

    -- Precio por m2 edificado, calculado a nivel de cada propiedad
    -- individual antes de promediar.
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno'
                    AND LOWER(moneda) = 'u$s'
                    AND LOWER(operacion) = 'venta'
                    AND mts2_edificado IS NOT NULL
                    AND mts2_edificado > 0
               THEN SAFE_DIVIDE(precio, mts2_edificado) END), 0) AS precio_x_m2_promedio_usd
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
WHERE
    LOWER(TRIM(detalles)) IN (
        'construcción sólida', 'chalet', 'casa independiente',
        'casa contenedor', 'cabaña', 'casa de piedra',
        'casa de barro', 'contenedor independiente'
    )
GROUP BY
    detalles
ORDER BY
    cantidad_propiedades DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_precio_tipo_construccion_completo = client.query(query_precio_tipo_construccion_completo).to_dataframe()

# Desplegamos el resultado.
df_precio_tipo_construccion_completo

,detalles,cantidad_propiedades,precio_promedio_usd,precio_x_m2_promedio_usd
0,Construcción sólida,46,290717.0,2066.0
1,Chalet,29,324828.0,2388.0
2,Casa independiente,5,173000.0,2145.0
3,Cabaña,3,183333.0,2748.0
4,Casa contenedor,3,194000.0,2011.0
5,Casa de piedra,2,585000.0,2492.0
6,Contenedor independiente,1,329000.0,1759.0
7,Casa de barro,0,NaN,NaN


### **Diagnóstico: Tipo de construcción vs. precio**

Categorías con muestra confiable (>5 casos):
1. "Chalet" sigue siendo la categoría con mejor precio por m2 dentro de este grupo (USD 2,388) y además tiene el precio total promedio más alto (USD 324,828) de las tres categorías robustas. Confirma su asociación con mayor categoría constructiva.
2. "Construcción sólida" es, con 46 casos, la etiqueta más usada del Dataset pero con el precio por m2 más bajo del grupo confiable (USD 2,066).
3. "Casa independiente" (5 casos) muestra el precio total más bajo (USD 173,000), aunque su precio por m2 (USD 2,145) es intermedio.

Categorías de referencia puntual (<5 casos, no representativas solo orientativas):

4. "Cabaña" (3 casos) tiene el valor por m2 más alto de TODA la tabla, incluyendo las categorías confiables: USD 2,748. Es un dato llamativo pero con solo 3 casos no se puede afirmar que las cabañas sean "el estilo más valioso", podría ser una coincidencia de 2-3 propiedades particulares bien ubicadas o de buen tamaño.
5. "Casa de piedra" (2 casos) tiene el precio total más alto de toda la tabla: USD 585,000 en promedio. De nuevo, con solo 2 propiedades, es un dato anecdótico, no hay forma de saber si el estilo "casa de piedra" es genuinamente premium o si son simplemente 2 propiedades caras que coinciden en tener esa etiqueta.
6. "Casa contenedor" tiene apenas 3 casos aquí (a pesar de que el conteo original de la columna detalles mostraba 17), la diferencia se explica porque la mayoría de esas 17 probablemente son alquileres. Vale la pena aclarar esto para que no parezca una inconsistencia entre preguntas.
7. "Casa de barro" no tiene ningún caso que cumpla los filtros (NaN en todo), la única propiedad con esa etiqueta en el Dataset no es una venta en USD, o no tiene los datos de precio/m2 necesarios.


---

### **Pregunta 6. Complejos y barrios privados vs. vivienda independiente**
Buscamos comparar el precio de las propiedades que están "En complejo", en "Barrio privado" o con "Club house con canchas y piscinas" frente a las viviendas independientes equivalentes, y cruzarlo con los gastos comunes, un costo recurrente que suele acompañar a este tipo de propiedades y que puede compensar (o no) un menor precio de compra. Como ya sabíamos, La Tahona es el barrio con mejor completitud de datos de gastos comunes y valores más altos, así que esperamos que sea el caso más claro para ilustrar esta relación.

In [ ]:
# =============================================================================
# COMPLEJOS/BARRIO PRIVADO vs. VIVIENDA INDEPENDIENTE (+ GASTOS COMUNES)
# =============================================================================

# Clasificamos cada propiedad como "Complejo/Privado" o "Independiente"
# según su detalle, y calculamos precio, $/m2 y gastos comunes promedio
# para cada grupo, tanto a nivel general como desglosado por barrio.
query_complejos_vs_independiente = """
WITH clasificacion AS (
    SELECT
        ubicacion AS barrio,
        precio,
        mts2_edificado,
        gastos_comunes_UYU,

        -- Agrupamos las categorías de "detalles" que implican vida en
        -- complejo/privado bajo una sola etiqueta, y todo lo demás
        -- (que no sea proximidad/amenity) como "Independiente".
        CASE
            WHEN LOWER(TRIM(detalles)) IN ('en complejo', 'barrio privado',
                 'club house con canchas y piscinas', 'complejo')
                THEN 'Complejo/Privado'
            ELSE 'Independiente'
        END AS categoria_vivienda
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
    WHERE
        LOWER(tipo_inmueble) != 'terreno'
        AND LOWER(moneda) = 'u$s'
        AND LOWER(operacion) = 'venta'
)

# Agregamos por barrio y por categoría (Complejo/Privado vs. Independiente),
# para comparar precio y gastos comunes entre ambos grupos, dentro de cada barrio.
SELECT
    barrio,
    categoria_vivienda,
    COUNT(*) AS cantidad_propiedades,
    ROUND(AVG(precio), 0) AS precio_promedio_usd,

    ROUND(AVG(CASE WHEN mts2_edificado IS NOT NULL AND mts2_edificado > 0
               THEN SAFE_DIVIDE(precio, mts2_edificado) END), 0) AS precio_x_m2_promedio_usd,

    -- Cantidad de propiedades con dato de gastos comunes cargado
    -- (para evaluar la completitud, ya que sabemos que varía por barrio).
    COUNTIF(gastos_comunes_UYU IS NOT NULL) AS cantidad_con_gastos_comunes,

    ROUND(AVG(gastos_comunes_UYU), 0) AS gastos_comunes_promedio_uyu
FROM
    clasificacion
GROUP BY
    barrio, categoria_vivienda
ORDER BY
    barrio, categoria_vivienda;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas.
df_complejos_vs_independiente = client.query(query_complejos_vs_independiente).to_dataframe()

# Desplegamos el resultado.
df_complejos_vs_independiente

,barrio,categoria_vivienda,cantidad_propiedades,precio_promedio_usd,precio_x_m2_promedio_usd,cantidad_con_gastos_comunes,gastos_comunes_promedio_uyu
0,Carrasco,Complejo/Privado,5,233600.0,2566.0,2,9500.0
1,Carrasco,Independiente,16,336913.0,2766.0,4,8850.0
2,Lagomar,Complejo/Privado,5,331600.0,3028.0,1,2000.0
3,Lagomar,Independiente,20,285000.0,2544.0,0,NaN
4,Pinar,Complejo/Privado,2,119500.0,2090.0,2,2600.0
5,Pinar,Independiente,16,261187.0,1601.0,0,NaN
6,Shangrila,Complejo/Privado,5,346000.0,3403.0,0,NaN
7,Shangrila,Independiente,27,294000.0,2277.0,1,5000.0
8,Solymar,Complejo/Privado,19,260368.0,2460.0,8,2213.0
9,Solymar,Independiente,164,278610.0,2296.0,6,3000.0


### **Diagnóstico: Complejos/Privado vs. Independiente**

1. La Tahona es el barrio con mejor completitud de gastos comunes. Con 13 de 16 casos "Complejo/Privado" (81%) y 11 de 14 "Independiente" (79%) con el dato cargado, es el único barrio donde este campo es confiable como para sacar conclusiones. El resto de los barrios tiene completitud muy baja o nula (Shangrilá-Complejo: 0 de 5; Lagomar-Independiente: 0 de 20; Pinar-Independiente: 0 de 16).
2. Los gastos comunes de La Tahona son entre 4x y 9x más altos que cualquier otro barrio con dato disponible, una diferencia abismal que confirma el perfil premium del barrio incluso en sus costos recurrentes, no solo en el precio de compra.
3. Hallazgo contraintuitivo: en La Tahona, "Complejo/Privado" es más BARATO que "Independiente" (USD 625,625 vs. USD 677,929), a pesar de tener gastos comunes prácticamente idénticos. Una explicación posible: las propiedades "Independientes" de La Tahona podrían ser de mayor tamaño o más exclusivas (casas de mayor superficie en lote propio), mientras que "Complejo/Privado" agrupa unidades más compactas dentro de un desarrollo compartido, el precio por m2 es similar (2,451 vs. 2,671), lo que sugiere que la diferencia de precio total viene más del tamaño que del formato en sí.
4. En el resto de los barrios el patrón es mixto, sin una tendencia clara. En Lagomar, Shangrilá y Pinar, "Complejo/Privado" es más caro que "Independiente"; en Carrasco y Solymar, es al revés.

Sin gastos comunes confiables para contrastar en la mayoría de estos casos, y con muestras chicas (2 a 5 propiedades en varias combinaciones) no se puede establecer un patrón general fuera de La Tahona.

**Advertencia de calidad de datos:** varias combinaciones tienen muestras muy chicas (Pinar-Complejo: 2, Carrasco-Complejo: 5, Lagomar-Complejo: 5), cualquier lectura sobre esos grupos específicos es orientativa.

La Tahona es el único caso del dataset donde se puede afirmar con confianza que el costo total de propiedad (precio de compra + gastos comunes recurrentes) es sustancialmente más alto que en el resto de la Costa, relevante para un comprador que evalúe no solo el precio de entrada sinó también el costo de mantenimiento a largo plazo.

Para el resto de los barrios, la falta de datos de gastos comunes impide sacar una conclusión sólida sobre el verdadero costo de vivir en complejo vs. independiente.

---

## **Cierre del Módulo:**
En este módulo cuantificamos el valor agregado de las características físicas de las propiedades: *amenities*, combinación de confort por segmento, habitaciones, proporción terreno/edificado, y categorías de la columna detalles (tipo de construcción y formato de complejo/privado), incluyendo su relación con gastos comunes en el caso de La Tahona.

#### **Con los 4 módulos académicos completos el proyecto pasa a su etapa de Business Intelligence: informes y visualizaciones orientados a la toma de decisiones empresariales.**

---
Análisis realizado en Julio/2026 con fines académicos.
---